# 2 — Is a supervised fault classifier viable?

**Or do we fall back to rule-based interpretation?**

Source: `protocols_recon.py`. Reads the visit protocols and the official variable
dictionary.

> ⚠️ **The outputs below are already saved — just scroll and read.**
> Do **not** press *Run*. This reads the 5 GB HEAPO dataset from a local folder
> that is not attached here, so re-running produces only `FileNotFoundError`.
> Everything you need to see is stored in the cells.

**Answer up front: no.** No fault type has enough labelled cases. The detail below
matters because it also proves the fault definitions are correct — all 16 rows of
the paper's Table 3 reconstruct to the exact published count.


In [1]:
import protocols_recon as P

pr  = P.load("protocols")
var = P.load("variables")
hh  = P.load("households")
q15 = P.load("15min")
print(f"protocols  {pr.shape[0]} rows x {pr.shape[1]} cols")
print(f"variables  {var.shape[0]} documented variables")

protocols  410 rows x 106 cols
variables  106 documented variables


## Step 1 — Inspection

Long by design. Every column profiled, the full variable dictionary printed, then a
list of findings that contradict the obvious assumptions — including the mixed
boolean encodings that have caused several bugs in this project.

In [2]:
P.step1_inspect(pr, var)

STEP 1 -- INSPECTION
protocols.csv shape: (410, 106)
protocols_variables.csv shape: (106, 6)

--- per-column profile ---
column                                                     dtype      non-null  uniq
Report_ID                                                  int64           410   410
Household_ID                                               float64         217   214
Visit_Year                                                 int64           410    10
Visit_Date                                                 str             304   233
Building_Type                                              str             409     3
Building_HousingUnits                                      float64         217     8
Building_ConstructionYear                                  float64         382    94
Building_ConstructionYear_Interval                         str             408     8
Building_Renovated_Windows                                 bool            410     2
Building_Renovated_Roof      

## Step 2 — Analysis base

Only rows with a household ID can be joined to smart meter data. 193 of 410 protocol
rows are anonymous.

In [3]:
base = P.step2_base(pr, hh)

STEP 2 -- ANALYSIS BASE
rows with non-null Household_ID: 217  (of 410)   distinct households: 214
expectation from the brief (217 rows / 214 households): MATCHES
households with multiple visits: 3  -> 3 extra rows beyond one-per-household
   household 120912: 2 visits, years [2020, 2023]
   household 111119: 2 visits, years [2021, 2023]
   household 8087988: 2 visits, years [2023, 2024]

households.csv Group=='treatment': 214
protocol base households:          214
in protocols but NOT treatment: 0 
treatment but NOT in protocols: 0 
=> the two sets are identical; the protocol base is exactly the treatment arm.



## Step 3 — Per-class counts, and Table 3 reconstruction

The reconstruction is the credibility test: if our fault definitions match the
paper's, the counts must match exactly.

In [4]:
label_counts = P.step3(pr, base, var)

STEP 3 -- PER-CLASS COUNTS
Counts are given on both frames: all 410 protocol rows (what the paper reports) and the
217-row analysis base that can actually be linked to a household.

--- listed candidates -- ALL protocol rows (n=410) ---
                                                 column        kind n_true n_false  n_null true_rate_nonnull                                    detail
       HeatPump_HeatingCurveSetting_TooHigh_BeforeVisit     boolean    168     207      35             0.448                                          
       HeatPump_HeatingLimitSetting_TooHigh_BeforeVisit     boolean    105     278      27            0.2742                                          
     HeatPump_NightSetbackSetting_Activated_BeforeVisit     boolean    172     215      23            0.4444                                          
                  DHW_TemperatureSetting_Categorization categorical                     16                         normal=362  too low=19  too high=13
        

## Step 4 — Label co-occurrence

In [5]:
P.step4(pr)

STEP 4 -- LABEL CO-OCCURRENCE
faults set simultaneously per protocol row (all 410):
  0 fault(s):   79 rows (19.3%)
  1 fault(s):   90 rows (22.0%)
  2 fault(s):  106 rows (25.9%)
  3 fault(s):   88 rows (21.5%)
  4 fault(s):   30 rows (7.3%)
  5 fault(s):   12 rows (2.9%)
  6 fault(s):    2 rows (0.5%)
  7 fault(s):    3 rows (0.7%)
  0 faults : 79
  1 fault  : 90
  2 faults : 106
  3+ faults: 135
  mean faults per row: 1.90   max: 7

top 6 faults by frequency: ['heating curve too high', 'night setback active', 'heating limit too high', 'descaling too long ago', 'expansion system', 'sizing (incorrectly planned)']

pairwise co-occurrence (count of rows where BOTH are set; diagonal = marginal):
                              heating curve too high  night setback active  heating limit too high  descaling too long ago  expansion system  sizing (incorrectly planned)
heating curve too high                         168.0                  82.0                    51.0                    34.0    

## Step 5 — Did the setting actually change?

In [6]:
P.step5(pr)

STEP 5 -- DID THE SETTING ACTUALLY CHANGE?
               setting  both_present  changed_observed    flag flag_true agree disagree flag_T_values_same flag_F_values_differ
 heating curve @ +20 C            58                15 Changed       192    40       17                 17                    0
   heating curve @ 0 C            56                29 Changed       192    54        1                  1                    0
  heating curve @ -8 C            65                37 Changed       192    62        2                  2                    0
         heating limit            88                69 Changed        79    88        0                  0                    0
         night setback           362               126  (none)                                                                 
       DHW temperature            32                32 Changed        34    32        0                  0                    0
circulation pump stage            22                22 Change

## Step 6 — Intersect with usable smart meter data

In [7]:
usable_tab = P.step6(base, q15, var)

STEP 6 -- INTERSECTION WITH USABLE SMART METER DATA (>= 180 d before AND after)
protocol base rows: 217  -> with 15min overview row: 217
rows meeting the >= 180 d both-sides rule: 89   distinct households: 89

--- listed candidates -- USABLE before/after sample (n=89) ---
                                                 column        kind n_true n_false  n_null true_rate_nonnull                                  detail
       HeatPump_HeatingCurveSetting_TooHigh_BeforeVisit     boolean     43      35      11            0.5513                                        
       HeatPump_HeatingLimitSetting_TooHigh_BeforeVisit     boolean     22      64       3            0.2558                                        
     HeatPump_NightSetbackSetting_Activated_BeforeVisit     boolean     30      57       2            0.3448                                        
                  DHW_TemperatureSetting_Categorization categorical                      2                          normal=79  too 

## Step 7 — Verdict

In [8]:
P.step7(usable_tab)

STEP 7 -- VERDICT
Thresholds used (stated so they can be argued with):
  (a) supervised classification : >= 50 positives AND >= 50 negatives in the usable before/after sample.
      Rationale: below ~50 minority cases a train/test split leaves single-digit test
      positives, and any performance estimate is dominated by split noise.
  (b) descriptive comparison    : >= 15 positives AND >= 15 negatives -- enough to compare group means, not to fit a model.
  (c) nothing                   : below (b).

                       fault  positives  negatives  minority_class                         verdict
      heating curve too high         43         46              43 (b) descriptive comparison only
      descaling too long ago         23         66              23 (b) descriptive comparison only
      heating limit too high         22         67              22 (b) descriptive comparison only
        night setback active         21         68              21 (b) descriptive comparison onl

## What this establishes

- Table 3 reconstructs **16 of 16 rows exactly**, once you use the right denominators
  (410 overall, 240 air-source, 167 ground-source). The fault definitions are correct.
- The faults are **multi-label, not multi-class**: 58.8% of visits carry two or more.
- Within the 89 usable households the largest fault class has **43 positives**.
- **No fault type supports supervised classification.** Four support descriptive
  comparison; twelve support nothing.
